# OpenBind blind challenge — run

Runtime > Change runtime type > **A100** (or L4), and tick **High-RAM** if offered.

What this notebook does differently from the first attempt, and why:
- the checkpoint and all output live on Colab's **local disk**, not the Drive mount,
  which is a network filesystem and was the likely cause of stalls
- output **streams live**, so a stall is visible immediately instead of after two hours
- each batch is **copied to Drive when it finishes**, so a disconnect costs one batch
- resume is worked out from what is actually on Drive, not from a log that can lie

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import psutil; print(f'{psutil.virtual_memory().total/1e9:.1f} GB RAM')

## 1. Drive and paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/openbind-blind-challenge')
DRIVE_PRED = ROOT / 'predictions'; DRIVE_PRED.mkdir(parents=True, exist_ok=True)
LOCAL = pathlib.Path('/content/preds'); LOCAL.mkdir(exist_ok=True)
print(len(list((ROOT/'data/queries').glob('batch_*.json'))), 'batch files found')

## 2. Install

In [ ]:
!pip -q install openfold3 awscli

## 3. Weights, copied to local disk

Downloads once to Drive, then copies to `/content` so inference never reads
a 2 GB file through the network mount.

In [ ]:
CKPT_DRIVE = ROOT / 'weights' / 'of3-ob-2025-06-30-174k.pt'
CKPT_DRIVE.parent.mkdir(exist_ok=True)
if not CKPT_DRIVE.exists():
    !aws s3 cp s3://openfold3-data/openfold3-parameters/of3-ob-2025-06-30-174k.pt "{CKPT_DRIVE}" --no-sign-request

CKPT = '/content/ckpt.pt'
!cp "{CKPT_DRIVE}" {CKPT}
!ls -lh {CKPT}

## 4. Runner config

Drop the `low_mem` lines if you are on an A100 — they cost speed and you do not
need them with 40 GB.

In [ ]:
(ROOT / 'runner.yml').write_text('''output_writer_settings:
  structure_format: pdb
model_update:
  presets:
    - predict
    - low_mem
''')
print((ROOT / 'runner.yml').read_text())

## 5. Smoke test — three compounds

Expect roughly 3 minutes. If this stalls, stop and read the output; do not go on.

In [ ]:
import json, time, pathlib

q = json.load(open(ROOT / 'data/queries/batch_000.json'))['queries']
three = {k: q[k] for k in list(q)[:3]}
pathlib.Path('/content/three.json').write_text(json.dumps({'queries': three}, indent=1))

t0 = time.time()
!run_openfold predict --query-json=/content/three.json --output-dir={LOCAL} --inference-ckpt-path={CKPT} --runner-yaml={ROOT}/runner.yml --num-diffusion-samples=1 --use-msa-server=True
print(f'\n{(time.time()-t0)/60:.1f} min for 3')

In [ ]:
outs = sorted(LOCAL.rglob('*_model.pdb'))
print(len(outs), 'structures')
for p in outs[:3]:
    het = sum(1 for l in p.read_text().splitlines() if l.startswith('HETATM'))
    print(p.name, het, 'ligand atoms')

## 6. The full run

Works through every batch, copying each to Drive as it completes, and skips
anything already saved there. Safe to re-run after a disconnect.

In [ ]:
import json, time

def saved():
    return {p.name for p in DRIVE_PRED.glob('blind_challenge_*')
            if any(p.rglob('*_model.pdb'))}

batches = sorted((ROOT / 'data/queries').glob('batch_*.json'))
print(f'{len(saved())} compounds already on Drive')

for b in batches:
    ids = list(json.load(open(b))['queries'])
    have = saved()
    if all(i in have for i in ids):
        print(f'{b.stem}: already done, skipping')
        continue
    print(f'\n===== {b.stem} =====', flush=True)
    t0 = time.time()
    !run_openfold predict --query-json={b} --output-dir={LOCAL} --inference-ckpt-path={CKPT} --runner-yaml={ROOT}/runner.yml --num-diffusion-samples=1 --use-msa-server=True
    !cp -r {LOCAL}/blind_challenge_* "{DRIVE_PRED}/"
    print(f'{b.stem}: {(time.time()-t0)/60:.1f} min, {len(saved())}/400 saved', flush=True)

## 7. Where are we

In [ ]:
have = sorted(p.name for p in DRIVE_PRED.glob('blind_challenge_*')
              if any(p.rglob('*_model.pdb')))
print(f'{len(have)}/400 complete')
missing = [f'blind_challenge_{i:03d}' for i in range(1, 401) if f'blind_challenge_{i:03d}' not in have]
print('missing:', missing[:20], '...' if len(missing) > 20 else '')